In [1]:
import re
import nltk
import pandas as pd
import spacy
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


path = 'C:/Users/Archana/OneDrive/Attachments/Desktop/MDTM46B/Project 6 - Hyperlocal News Anamoly Detection/Articles.csv'
df = pd.read_csv(path, encoding='latin1')
df = df.drop_duplicates(subset=['Article','Heading'])
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Article']).reset_index(drop=True)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Archana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Archana\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Archana\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
df.isna().sum()

Article     0
Date        0
Heading     0
NewsType    0
dtype: int64

In [3]:
import os, re, warnings
warnings.filterwarnings('ignore')

In [4]:


df = df.drop_duplicates(subset=['Article','Heading'])
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Article']).reset_index(drop=True)

In [5]:
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_locations(text):
    doc = nlp(str(text))
    location= list({ent.text for ent in doc.ents if ent.label_ == "GPE"})
    return list(dict.fromkeys(location))

df['location'] = df['Article'].apply(extract_locations)


In [6]:

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)


df['Clean_Article'] = df['Article'].apply(clean_text)
df['word_count'] = df['Clean_Article'].apply(lambda x: len(x.split()))
df['year']  = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day']   = df['Date'].dt.day
df['dayofweek'] = df['Date'].dt.dayofweek

In [ ]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")
def get_sentiment(text):
    text = str(text)[:512]   
    result = sentiment_model(text)[0]
    return pd.Series([result['label'], result['score']])


df[['sentiment', 'sentiment_score']] = df['Article'].apply(get_sentiment)
label_map = {'LABEL_0': 'Negative', 'LABEL_1': 'Neutral', 'LABEL_2': 'Positive'}
df['sentiment'] = df['sentiment'].map(label_map)

In [ ]:
# 6. Sentence Embeddings

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(df['Clean_Article'].tolist(), show_progress_bar=True, convert_to_numpy=True)

print("Embeddings shape:", embeddings.shape)

In [ ]:
from bertopic import BERTopic
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=False)
topics, probs = topic_model.fit_transform(df['Clean_Article'].tolist(), embeddings)
df['Topic'] = topics
df['topic_probabilities'] = [None if p is None else p.tolist() for p in probs]

In [ ]:
topic_info = topic_model.get_topic_info()
top_topic_map = {}
for t in topic_info.Topic:
    if t == -1:
        top_topic_map[-1] = "Outlier"
    else:
        words = topic_model.get_topic(t)
        label = " ".join([w for w, _ in words[:3]]) if words else f"Topic_{t}"
        top_topic_map[t] = label
df['topic_label'] = df['Topic'].map(top_topic_map)

In [ ]:
NEWS_KEYWORDS = {
    "crime": ["murder", "robbery", "police", "arrest", "crime", "stabbing", "killed", "shooting", "investigation"],
    "festival": ["festival", "celebration", "celebrate", "parade", "fete", "garland", "pandal", "ceremony"],
    "business": ["company", "business", "market", "shares", "stock", "acquisition", "revenue", "profit"],
    "sports": ["match", "score", "tournament", "league", "goal", "cricket", "football", "olympics", "player"],
    "politics": ["election", "minister", "government", "policy", "parliament", "mp", "politics", "campaign"],
    "technology": ["tech", "startup", "ai", "machine learning", "software", "app", "gadget", "device"],
}

def auto_classify_newstype(text):
    txt = str(text).lower()
    counts = {cat: sum(1 for kw in kws if kw in txt) for cat, kws in NEWS_KEYWORDS.items()}
    counts = {k:v for k,v in counts.items() if v>0}
    if counts:
        return sorted(counts.items(), key=lambda x: (-x[1], x[0]))[0][0]
    return "other"

df['NewsType_auto'] = df['Clean_Article'].apply(auto_classify_newstype)
df['NewsType_final'] = df.apply(lambda r: r['NewsType_auto'], axis=1)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()
X_scaled = scaler.fit_transform(embeddings)


iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X_scaled)
anomaly_pred = iso.predict(X_scaled)
anomaly_score = iso.decision_function(X_scaled)

df['anomaly_pred'] = anomaly_pred
df['anomaly_score'] = anomaly_score
df['is_anomaly'] = df['anomaly_pred'] == -1


In [ ]:
df['main_location']= df['location'].str[0]
df.to_csv("cleaned_data.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split


loc_encoder = LabelEncoder()
df['loc_encoded'] = loc_encoder.fit_transform(df['main_location'])

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(df['Clean_Article'].astype(str).tolist())

# Combine embeddings + location
X = np.concatenate([embeddings, df['loc_encoded'].values.reshape(-1, 1)], axis=1)
y = df['is_anomaly']  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)


clf = XGBClassifier()
clf.fit(X_train_scaled,np.ravel(y_train))


# Save models
scaler_path="scaler1.pkl"
scaler.save_model(scaler_path)
print(f"Scaler Model save to {scaler_path}")
filepath="XGBClassifier1.json" 
clf.save_model(filepath)
print(f"Model save to {filepath}")
label_enc_path="LabelEncoder1.pkl"
loc_encoder.save_model(label_enc_path)
print(f"Label encoder Model save to {label_enc_path}")


In [ ]:
X_train.shape

In [ ]:
X_train_scaled.shape

In [ ]:
from xgboost import XGBClassifier

clf = XGBClassifier(eval_metric='logloss', random_state=42, use_label_encoder=False)
clf.fit(embeddings, df['loc_encoded'])
preds = clf.predict_proba(embeddings)
df['loc_pred'] = preds.argmax(axis=1)
df['loc_confidence'] = preds.max(axis=1)
df['loc_pred_name'] = loc_encoder.inverse_transform(df['loc_pred'])

filepath="XGBClassifier.json" 
clf.save_model(filepath)
print(f"Model save to {filepath}")

In [ ]:
preds.shape

In [ ]:
df


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

In [ ]:

df['anomaly_prob'] = -df['anomaly_score']


In [ ]:
y_true = df['is_anomaly'].astype(int)
y_pred = (df['anomaly_prob'] > np.percentile(df['anomaly_prob'], 95)).astype(int)

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, df['anomaly_prob'])

# Recall@K (Top 5%)
k = int(0.05 * len(df))
top_k = df.nlargest(k, 'anomaly_prob')
recall_at_k = top_k['is_anomaly'].mean()

print("/n Evaluation Metrics (Unsupervised Simulation):")
print(f"AUC: {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Recall@5%: {recall_at_k:.4f}")